# Análise RFM — FiberNet ISP
### Segmentação comportamental de clientes por padrão de pagamento

**Hugo Nazário · Analista de Dados Pleno · Speed Fibra**

---

**Pergunta de negócio:** Dos 300 contratos ativos na base FiberNet, quais clientes são
os mais valiosos, quais estão em risco silencioso e quais já estão efetivamente perdidos?

**Por que RFM para ISP?**  
Diferente de e-commerce, onde RFM analisa compras, em ISP analisamos *pagamentos de boleto*.
O padrão de pagamento é o sinal mais precoce de churn: o cliente para de pagar antes de pedir o cancelamento.
Identificar queda na recência ou frequência com 30-60 dias de antecedência permite ação preventiva
com custo muito menor do que reativar um cliente já cancelado.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

REFERENCE_DATE = pd.Timestamp('2024-10-15')
SEED = 42

print('Bibliotecas carregadas. Data de referência:', REFERENCE_DATE.date())

---
## 1. Carregamento dos dados

Os dados vêm do banco FiberNet ISP (`sql-analytics-pack`): 300 contratos, ~4.241 boletos,
período jan/2022–out/2024.

**Execute a seção A (PostgreSQL) ou B (Standalone) conforme seu ambiente.**

In [ ]:
# ── SEÇÃO A: PostgreSQL ────────────────────────────────────────────────────────
# Requer sql-analytics-pack carregado em fibernet_analytics
# Comente este bloco e execute o Bloco B se não tiver banco disponível

USE_DB = False  # Mude para True se tiver PostgreSQL disponível

if USE_DB:
    from sqlalchemy import create_engine, text
    engine = create_engine('postgresql://postgres:@localhost/fibernet_analytics')

    with engine.connect() as conn:
        df_clients = pd.read_sql(text("""
            SELECT
                c.id          AS client_id,
                c.city,
                c.status,
                p.name        AS plan,
                ct.amount     AS monthly_amount
            FROM clients c
            JOIN contracts ct ON ct.client_id = c.id
            JOIN plans     p  ON p.id = ct.plan_id
            WHERE ct.status = 'active'
               OR ct.cancellation_date IS NOT NULL
        """), conn)

        df_receivables = pd.read_sql(text("""
            SELECT
                ct.client_id,
                fr.paid_at,
                fr.amount
            FROM financial_receivables fr
            JOIN contracts ct ON ct.id = fr.contract_id
            WHERE fr.paid_at IS NOT NULL
        """), conn)

    print(f'Clientes carregados: {len(df_clients)}')
    print(f'Boletos pagos carregados: {len(df_receivables)}')

In [ ]:
# ── SEÇÃO B: Standalone (dados sintéticos FiberNet ISP) ───────────────────────
# Executa sempre que USE_DB = False

if not USE_DB:
    rng = np.random.default_rng(SEED)

    PLANS = {
        'Fibra 100MB':  89.0,
        'Fibra 200MB': 119.0,
        'Fibra 500MB': 149.0,
        'Empresarial': 449.0,
    }
    PLAN_CHURN = {'Fibra 100MB': 0.367, 'Fibra 200MB': 0.25, 'Fibra 500MB': 0.18, 'Empresarial': 0.10}
    CITIES   = ['Betim', 'Contagem', 'Ribeirão das Neves', 'Esmeraldas', 'Ibirité']
    CITY_W   = [0.274, 0.240, 0.200, 0.160, 0.126]
    PLAN_W   = [0.35, 0.30, 0.25, 0.10]
    N        = 300
    START    = pd.Timestamp('2022-01-01')

    plan_names   = rng.choice(list(PLANS.keys()), size=N, p=PLAN_W)
    city_names   = rng.choice(CITIES, size=N, p=CITY_W)
    start_days   = rng.integers(0, (REFERENCE_DATE - START).days, size=N)
    start_dates  = START + pd.to_timedelta(start_days, unit='D')
    tenure_months = np.maximum(1, ((REFERENCE_DATE - start_dates) / pd.Timedelta(days=30)).astype(int))

    cancelled = rng.random(N) < np.array([PLAN_CHURN[p] for p in plan_names])
    # Cancelled clients left at a random point in their tenure
    cancel_at = np.where(
        cancelled,
        np.maximum(1, (tenure_months * rng.uniform(0.3, 0.9, N)).astype(int)),
        tenure_months,
    )

    df_clients = pd.DataFrame({
        'client_id':      range(1, N + 1),
        'city':           city_names,
        'plan':           plan_names,
        'monthly_amount': [PLANS[p] for p in plan_names],
        'status':         np.where(cancelled, 'cancelled', 'active'),
        'start_date':     start_dates,
        'active_months':  cancel_at,
    })

    # Generate invoice records (one per month active)
    rows = []
    for _, c in df_clients.iterrows():
        for m in range(int(c['active_months'])):
            due = c['start_date'] + pd.DateOffset(months=m + 1)
            # ~8% late payment probability; late by 5-20 days
            late = rng.random() < 0.08
            paid_offset = rng.integers(1, 5) if not late else rng.integers(6, 21)
            paid_at = due + pd.Timedelta(days=int(paid_offset))
            if paid_at > REFERENCE_DATE:
                paid_at = None  # unpaid (future or overdue at reference date)
            amount = c['monthly_amount'] * rng.uniform(0.98, 1.02)
            rows.append({
                'client_id': c['client_id'],
                'due_date':  due,
                'paid_at':   paid_at,
                'amount':    round(amount, 2),
            })

    df_receivables = pd.DataFrame(rows)
    paid_mask = df_receivables['paid_at'].notna()

    print(f'Clientes gerados  : {len(df_clients)} ({cancelled.sum()} cancelados)')
    print(f'Boletos gerados   : {len(df_receivables)}')
    print(f'Boletos pagos     : {paid_mask.sum()}')
    print(f'MRR ativo (aprox) : R$ {df_clients[df_clients["status"]=="active"]["monthly_amount"].sum():,.0f}')

---
## 2. Calculando R, F, M para cada cliente

A unidade de análise é o **cliente**. Para cada um calculamos:

- **R (Recência):** quantos dias se passaram desde o último boleto pago até `2024-10-15`
- **F (Frequência):** quantos boletos foram pagos no total
- **M (Monetário):** soma total de todos os boletos pagos (valor acumulado do cliente)

Somente boletos com `paid_at` preenchido entram no cálculo.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))
from rfm import calculate_rfm

df_rfm = calculate_rfm(
    df_clients=df_clients,
    df_receivables=df_receivables,
    reference_date=REFERENCE_DATE,
)

print(f'Clientes com ao menos 1 boleto pago: {len(df_rfm)}')
print(f'\nEstatísticas descritivas:')
df_rfm[['recency_days', 'frequency', 'monetary']].describe().round(1)

In [ ]:
# Distribuição dos valores brutos — antes de qualquer scoring
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Recência (dias)', 'Frequência (boletos pagos)', 'Monetário (R$ acumulado)'),
)

for col_i, (col, label) in enumerate(
    [('recency_days', 'Dias'), ('frequency', 'Boletos'), ('monetary', 'R$')], start=1
):
    fig.add_trace(
        go.Histogram(x=df_rfm[col], nbinsx=20, name=label,
                     marker_color=['#4f8ef7', '#10b981', '#f59e0b'][col_i - 1],
                     showlegend=False),
        row=1, col=col_i,
    )

fig.update_layout(
    title='Distribuição das métricas RFM brutas — FiberNet ISP',
    height=350,
    paper_bgcolor='#f8fafc',
    plot_bgcolor='#f8fafc',
)
fig.show()

print('\nObservação:')
print(f'  Recência mediana : {df_rfm["recency_days"].median():.0f} dias')
print(f'  Frequência mediana: {df_rfm["frequency"].median():.0f} boletos')
print(f'  Monetário mediano : R$ {df_rfm["monetary"].median():,.0f}')

**O que vemos:** A distribuição de recência é bimodal — há um grupo com pagamentos muito recentes
(clientes ativos em dia) e outro com recência longa (cancelados ou inadimplentes crônicos).
Isso é exatamente o que esperamos: o ISP tem uma base mista de ativos e em evasão.

---
## 3. Transformando em scores comparáveis (quintis 1–5)

Valores brutos não são comparáveis entre si: 30 dias de recência é bom;
R$ 5.000 acumulado pode ser bom ou ruim dependendo do plano.

A solução é **quintis**: dividir cada métrica em 5 grupos de tamanho igual (20% cada).
Score 5 = melhor 20% da base; Score 1 = pior 20%.

Atenção: para Recência, **menor número de dias = melhor** → score 5 para quem pagou mais recentemente.

In [ ]:
from rfm import score_rfm

df_scored = score_rfm(df_rfm)

print('Distribuição dos scores por quintil:')
for col in ['r_score', 'f_score', 'm_score']:
    dist = df_scored[col].value_counts().sort_index()
    print(f'\n  {col}:')
    for score, count in dist.items():
        bar = '█' * int(count / len(df_scored) * 40)
        print(f'    {score}: {bar} ({count} clientes, {count/len(df_scored)*100:.1f}%)')

---
## 4. Segmentação: agrupando clientes por comportamento

Com os três scores, podemos identificar padrões comportamentais que mapeiam diretamente
para ações comerciais. A segmentação combina R, F e M via regras de negócio:

- **Campeões (R5, F4-5, M4-5):** pagam em dia, pagam muito, pagam há muito tempo
- **Em Risco (R2-3, F3-5, M3-5):** bom histórico mas recência caindo — pré-churn
- **Hibernando (R1-2, F1-3, M1-3):** sem atividade recente significativa
- **Perdidos (R1, F1-2, M1-2):** provavelmente já cancelados ou sem contato

In [ ]:
from rfm import assign_segments, segment_summary

df_segmented = assign_segments(df_rfm)

summary = segment_summary(df_segmented)
print('Resumo por segmento:\n')
print(summary[['segment', 'clientes', 'pct_clientes', 'mrr_total', 'pct_mrr', 'recency_media', 'freq_media']]
      .to_string(index=False))

---
## 5. Visualizações

### 5.1 Heatmap: distribuição R × F por score

In [ ]:
heat_data = (
    df_segmented.groupby(['r_score', 'f_score'])
    .size()
    .unstack(fill_value=0)
)

fig = go.Figure(data=go.Heatmap(
    z=heat_data.values,
    x=[f'F={c}' for c in heat_data.columns],
    y=[f'R={i}' for i in heat_data.index],
    colorscale='Blues',
    text=heat_data.values,
    texttemplate='%{text}',
    hovertemplate='R=%{y}, F=%{x}<br>Clientes: %{z}<extra></extra>',
))
fig.update_layout(
    title='Heatmap R × F — número de clientes por combinação de score',
    xaxis_title='Score de Frequência (5 = mais frequente)',
    yaxis_title='Score de Recência (5 = mais recente)',
    height=450,
    paper_bgcolor='#f8fafc',
)
fig.show()

print('Concentração ideal: quadrante R5×F5 (canto superior direito) = Campeões')
print('Alerta: clientes R1-2 com F4-5 = tinham alta frequência mas sumiram = Em Risco crítico')

### 5.2 Scatter: Recência × Monetário colorido por Frequência

In [ ]:
fig = px.scatter(
    df_segmented,
    x='recency_days',
    y='monetary',
    color='f_score',
    color_continuous_scale='Blues',
    hover_data=['client_id', 'frequency', 'segment'],
    labels={
        'recency_days': 'Recência (dias desde último pagamento)',
        'monetary':     'Valor Monetário Acumulado (R$)',
        'f_score':      'Score Frequência',
    },
    title='Recência × Monetário — colorido por Score de Frequência',
    height=500,
)
# Highlight Em Risco segment
em_risco = df_segmented[df_segmented['segment'] == 'Em Risco']
if len(em_risco):
    fig.add_trace(go.Scatter(
        x=em_risco['recency_days'], y=em_risco['monetary'],
        mode='markers',
        marker=dict(symbol='circle-open', size=14, color='#ef4444', line=dict(width=2)),
        name='Em Risco (destaque)',
        hovertemplate='ID: %{customdata[0]}<br>R: %{x}d M: R$%{y:,.0f}',
        customdata=em_risco[['client_id']].values,
    ))
fig.update_layout(paper_bgcolor='#f8fafc', plot_bgcolor='#f8fafc')
fig.show()

print('Clientes Em Risco destacados: alta frequência histórica + recência caindo')
print(f'  Recência média Em Risco: {em_risco["recency_days"].mean():.0f} dias')
print(f'  Monetário médio Em Risco: R$ {em_risco["monetary"].mean():,.0f}')

### 5.3 Treemap: tamanho de cada segmento × MRR

In [ ]:
SEG_COLORS = {
    'Campeões':        '#1d4ed8',
    'Leais':           '#2563eb',
    'Potenciais Leais':'#60a5fa',
    'Em Risco':        '#ef4444',
    'Hibernando':      '#f97316',
    'Perdidos':        '#6b7280',
    'Outros':          '#9ca3af',
}

summary_plot = summary.copy()
summary_plot['cor'] = summary_plot['segment'].map(SEG_COLORS).fillna('#9ca3af')
summary_plot['label'] = (
    summary_plot['segment'] + '<br>' +
    summary_plot['clientes'].astype(str) + ' clientes<br>' +
    'R$ ' + summary_plot['mrr_total'].apply(lambda x: f'{x:,.0f}')
)

fig = go.Figure(go.Treemap(
    labels=summary_plot['label'],
    parents=[''] * len(summary_plot),
    values=summary_plot['clientes'],
    customdata=summary_plot[['mrr_total', 'pct_mrr']].values,
    hovertemplate=(
        '<b>%{label}</b><br>'
        'MRR: R$ %{customdata[0]:,.0f}<br>'
        '% MRR: %{customdata[1]:.1f}%<extra></extra>'
    ),
    marker_colors=summary_plot['cor'],
    textfont_size=13,
))
fig.update_layout(
    title='Treemap de segmentos — tamanho proporcional ao número de clientes',
    height=500,
)
fig.show()

### 5.4 MRR por segmento — onde está a receita

In [ ]:
fig = go.Figure()
seg_order = ['Campeões', 'Leais', 'Potenciais Leais', 'Em Risco', 'Hibernando', 'Perdidos', 'Outros']
plot_df = summary.set_index('segment').reindex([s for s in seg_order if s in summary['segment'].values])

colors = [SEG_COLORS.get(s, '#9ca3af') for s in plot_df.index]

fig.add_trace(go.Bar(
    y=plot_df.index,
    x=plot_df['mrr_total'],
    orientation='h',
    marker_color=colors,
    text=plot_df['pct_mrr'].apply(lambda x: f'{x:.1f}% do MRR'),
    textposition='outside',
    hovertemplate='%{y}<br>MRR: R$ %{x:,.0f}<extra></extra>',
))
fig.update_layout(
    title='MRR por segmento — concentração de receita',
    xaxis_title='MRR (R$)',
    height=400,
    paper_bgcolor='#f8fafc',
    plot_bgcolor='#f8fafc',
    showlegend=False,
)
fig.show()

---
## 6. O que os dados dizem para o negócio

In [ ]:
total_mrr = summary['mrr_total'].sum()
total_clientes = summary['clientes'].sum()

defensores = summary[summary['segment'].isin(['Campeões', 'Leais'])]
ameacados  = summary[summary['segment'].isin(['Em Risco', 'Hibernando'])]

pct_clientes_def = defensores['clientes'].sum() / total_clientes * 100
pct_mrr_def      = defensores['mrr_total'].sum() / total_mrr * 100
mrr_ameacado     = ameacados['mrr_total'].sum()
pct_mrr_ameac    = mrr_ameacado / total_mrr * 100

em_risco_row  = summary[summary['segment'] == 'Em Risco'].iloc[0] if 'Em Risco' in summary['segment'].values else None
hibernando_row = summary[summary['segment'] == 'Hibernando'].iloc[0] if 'Hibernando' in summary['segment'].values else None

print('=' * 60)
print('RESUMO EXECUTIVO — RFM FiberNet ISP')
print('=' * 60)
print(f'Base total analisada : {total_clientes} clientes')
print(f'MRR total (histórico): R$ {total_mrr:,.0f}')
print()
print('CONCENTRAÇÃO DE RECEITA:')
print(f'  Campeões + Leais = {pct_clientes_def:.0f}% dos clientes')
print(f'                   = {pct_mrr_def:.0f}% do MRR — base da operação')
print()
print('MRR EM RISCO:')
print(f'  Em Risco + Hibernando = R$ {mrr_ameacado:,.0f} ({pct_mrr_ameac:.1f}% do MRR)')
if em_risco_row is not None:
    print(f'  → Em Risco : {int(em_risco_row["clientes"])} clientes, '
          f'R$ {em_risco_row["mrr_total"]:,.0f}/mês — ação urgente (30 dias)')
if hibernando_row is not None:
    print(f'  → Hibernando: {int(hibernando_row["clientes"])} clientes, '
          f'R$ {hibernando_row["mrr_total"]:,.0f}/mês — campanha de reativação')
print()
print('RETENÇÃO DE 50% DOS "EM RISCO" VALE:')
if em_risco_row is not None:
    retencao_anual = em_risco_row['mrr_total'] * 0.5 * 12
    print(f'  R$ {retencao_anual:,.0f}/ano em MRR preservado')
print('=' * 60)

---
## 7. Recomendações por segmento

A análise RFM só tem valor se conectada a ação. Cada segmento tem uma ação diferente —
e tratar todos os clientes da mesma forma desperdiça recursos comerciais.

In [ ]:
from rfm import SEGMENT_RULES

print(f'{'Segmento':<20} {'Clientes':>9} {'MRR (R$)':>11}  Ação')
print('-' * 100)
for rule in SEGMENT_RULES:
    seg_name = rule['segment']
    row = summary[summary['segment'] == seg_name]
    if len(row):
        r = row.iloc[0]
        mrr_str = f"R$ {r['mrr_total']:,.0f}"
        print(f'{seg_name:<20} {int(r["clientes"]):>9}  {mrr_str:>11}  {rule["action"]}')
    else:
        print(f'{seg_name:<20} {"—":>9}  {"—":>11}  {rule["action"]}')

In [ ]:
# Exportar lista de clientes Em Risco para ação imediata
em_risco_clientes = df_segmented[df_segmented['segment'] == 'Em Risco'].copy()
em_risco_clientes = em_risco_clientes.sort_values('monetary', ascending=False)

os.makedirs('../outputs', exist_ok=True)
em_risco_clientes.to_csv('../outputs/em_risco_acao_imediata.csv', index=False, encoding='utf-8-sig')

print(f'Arquivo exportado: outputs/em_risco_acao_imediata.csv')
print(f'Clientes em risco: {len(em_risco_clientes)}')
if 'monthly_amount' in em_risco_clientes.columns:
    print(f'MRR ameaçado    : R$ {em_risco_clientes["monthly_amount"].sum():,.0f}')
print(f'\nTop 5 por valor acumulado:')
cols_show = ['client_id', 'recency_days', 'frequency', 'monetary', 'r_score', 'f_score', 'm_score']
if 'plan' in em_risco_clientes.columns:
    cols_show.insert(1, 'plan')
if 'city' in em_risco_clientes.columns:
    cols_show.insert(2, 'city')
print(em_risco_clientes[cols_show].head(5).to_string(index=False))

---
## 8. Próximos passos

1. **Integrar com churn-predictor:** cruzar segmento RFM com probabilidade de churn do modelo XGBoost.
   Clientes `Em Risco` com P(churn) > 0.6 são prioridade crítica.

2. **Análise temporal:** repetir o cálculo mensalmente e rastrear *migrações* de segmento
   (ex: quantos `Campeões` viraram `Em Risco` nos últimos 3 meses?).

3. **RFM por plano:** o comportamento de `Fibra 100MB` (churn 36.7%) deve diferir
   de `Empresarial` — segmentar separadamente para ações comerciais distintas.

4. **Conectar ao NOC/SLA:** clientes `Em Risco` com tickets abertos têm probabilidade
   de churn multiplicada. Cruzar com dados do `telecom-kpi-dashboard`.

---
*Hugo Nazário · Analista de Dados Pleno · Speed Fibra · out/2024*